In [1]:
import sys
sys.path.append('../../BasicDFIR/')
import yaml
from basicsr.archs.jit_arch import *
from basicsr.utils.options import *
from basicsr.models import build_model
from basicsr.data import build_dataloader, build_dataset
from basicsr.data.data_sampler import EnlargedSampler
from matplotlib import pyplot as plt
import numpy as np

/home/ybb/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 定义模型

In [6]:
opt = yaml_load('../options/train/SwinIR/train_SwinIR_S_SRx2_scratch_B8_P32.yml')
opt['path']['pretrain_network_g'] = '/8T2/Project/BasicDFIR/experiments/train_SwinIR_S_SRx2_scratch_B8_P32_20260204_102017_835836/models/net_g_140000.pth'
opt['strict_load_g'] = 'true'
opt['param_key_g']= 'params_ema'
opt['is_train']=0
opt['dist']= False
opt['rank'], opt['world_size']=0, 1
model = build_model(opt)

In [7]:
torch.mean(list(model.net_g.parameters())[0]) #测试加载参数是否成功

tensor(-0.0004, device='cuda:0', grad_fn=<MeanBackward0>)

### 定义数据集

In [9]:
dataset_opt = opt['datasets']['val_1']
dataset_opt['phase'] ='val'
dataset_opt['scale'] =2
dataset_enlarge_ratio = 1
test_set = build_dataset(dataset_opt)
test_sampler = EnlargedSampler(test_set, opt['world_size'], opt['rank'], dataset_enlarge_ratio)
test_loader = build_dataloader(
    test_set,
    dataset_opt,
    num_gpu=opt['num_gpu'],
    dist=opt['dist'],
    sampler=test_sampler,
    seed=opt['manual_seed'])

In [20]:
model.opt['path']['visualization']= './'
model.opt['val']['suffix']=''

In [21]:
model.validation(test_loader, current_iter=opt['name'], tb_logger=None, save_img=True)
model.metric_results

{'psnr': np.float64(37.77157070912116),
 'ssim': np.float64(0.9599723840590979),
 'lpips': 0.057004056964069606}

In [12]:
model.metric_results

{'psnr': np.float64(37.77157070912116),
 'ssim': np.float64(0.9599723840590979),
 'lpips': 0.057004056964069606}

In [41]:
iter_dataloader = iter(train_loader)

In [46]:
data = next(iter_dataloader)
lq, gt = data['lq'], data['gt']
print(data['lq_path'])

['/8T1/dataset/SRDataset/ImageSR/test/Set5/image_SRF_2/LR/img_001.png']
